# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-09 · Audits both the published FlyRank paper
> and my own ML-08 model. Constructive on both counts — the goal is the next level of rigour,
> not a scalp.

In [1]:
%pip install -q pandas scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, subprocess, sys
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 20260808
REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT = REPO / "work/outputs"
rel = lambda q: q.relative_to(REPO).as_posix()

for name in ["dev", "sealed"]:
    if not (OUT / f"modeling_frame_{name}.parquet").exists():
        subprocess.run([sys.executable, str(REPO / "work/scripts/build_modeling_frame.py"),
                        "--sealed"], check=True)
        break

dev    = pd.read_parquet(OUT / "modeling_frame_dev.parquet").reset_index(drop=True)
sealed = pd.read_parquet(OUT / "modeling_frame_sealed.parquet").reset_index(drop=True)
m5     = json.load(open(OUT / "w05_model_metrics.json"))

print(f"dev    : {len(dev):,} items · {dev.client_hash_id.nunique()} clients · "
      f"base {dev.is_position_decline.mean():.4f}  (features Jan-Mar 2026 -> label Apr)")
print(f"sealed : {len(sealed):,} items · {sealed.client_hash_id.nunique()} clients · "
      f"base {sealed.is_position_decline.mean():.4f}  (features Mar-May 2026 -> label Jun)")

dev    : 106,461 items · 42 clients · base 0.5673  (features Jan-Mar 2026 -> label Apr)
sealed : 116,656 items · 48 clients · base 0.5622  (features Mar-May 2026 -> label Jun)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Both questions below are about **design**, not about whether the authors are right. The paper is
unusually candid — it flags its own unstable buckets, states "correlations do not prove
causation", and demotes its ML work to an appendix. These are the two places where I think the
words still travel further than the design carries them.

---

### Finding #4 — "The Freshness Multiplier" (p. 9)

> *"365+ day content that was refreshed within 30 days shows 3.2× health boost (from 10.7 to
> 34.5) and 57× more impressions (from 71 to 4039)… refresh timing is one of the strongest
> measured levers available."*

**Where does the label come from?** "Refreshed within 30 days" is not an assigned treatment —
it is a record of **what an SEO team chose to do**. That choice was almost certainly not random:
teams refresh pages they believe are worth saving, which usually means pages with existing
visibility, commercial value, or a recent internal flag. The comparison group ("untouched
mature pages") therefore contains the pages nobody thought were worth the effort.

**Does the validation design carry the claim?** The number the design supports is *"refreshed
mature pages were observed at 57× the impressions of unrefreshed ones."* The word **"lever"**,
and the recommended action *"Expected: Lifts mature pages from roughly 10.7 health to 34.5"*,
promise that **pulling** the lever produces the gap. A selection effect and a treatment effect
are indistinguishable here, and 57× is far larger than any plausible refresh effect — which is
itself evidence that most of the gap is selection.

**A second, separable problem: the two headline numbers are not independent.** Health Score is
defined on p. 5 as impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll (20 pts).
Impressions are **30% of health score by construction**. So "3.2× health *and* 57× impressions"
is closer to one finding reported twice than two corroborating findings.

**How to make it stronger (cheap, no new data):** match refreshed and unrefreshed mature pages
on their **pre-refresh** impressions, position, and age, then compare *post-refresh change*
between matched pairs. That removes most of the selection effect. Reporting the pre-period
distributions of both groups alone would let a reader judge the size of the problem.

---

### Finding: the ML appendix's split (Methodology, p. 36)

> *"Sklearn: K-Means (k=5), Random Forest (80/20 split), Logistic Regression (80/20 split),
> PCA (2 components), and a shallow Decision Tree (80/20 split)."*

**My question:** was the 80/20 split **grouped by brand**? The paper spans 57 brands and 341,701
pages, so a plain random 80/20 puts most brands on both sides of the split. A model can then
identify the brand and recite its average outcome instead of learning a content signal.

**This is not a hypothetical objection — I measured it on the same kind of data**, and the
result is in the table below: my random forest scores **AUC 0.776 / precision@50 = 1.000** under
a random split and **AUC 0.649 / precision@50 = 0.800** under a client-grouped split on
identical rows. The random split roughly **doubled** my apparent skill above base rate.

**How to make it stronger:** report the ML appendix numbers under `GroupKFold` on brand, or
state explicitly that the split was random and treat the figures as an upper bound. Given the
paper already demotes ML to "exploratory", one sentence naming the split would settle it.

In [3]:
# The evidence behind my question about the paper's split — my own numbers, same rows.
gap = pd.DataFrame({
    "grouped_by_client": {k: m5["grouped"][k]["roc_auc"] for k in
                          ["logistic_regression", "decision_tree_d4", "random_forest"]},
    "random_split":      m5["random_split_diagnostic"],
})
gap["inflation"] = gap.random_split - gap.grouped_by_client
print("ROC AUC — same data, same models, only the split differs:\n")
print(gap.round(3).to_string())
print(f"\nrandom forest precision@50: grouped {m5['grouped']['random_forest']['P@50']:.3f} "
      f"vs random split 1.000")
print(f"base rate {m5['base_rate']:.3f}")
print("\nSkill above base rate is roughly halved by grouping. That is the size of the")
print("question I am asking of the paper's 80/20 split.")

ROC AUC — same data, same models, only the split differs:

                     grouped_by_client  random_split  inflation
logistic_regression              0.617         0.682      0.065
decision_tree_d4                 0.612         0.676      0.065
random_forest                    0.651         0.777      0.126

random forest precision@50: grouped 0.880 vs random split 1.000
base rate 0.567

Skill above base rate is roughly halved by grouping. That is the size of the
question I am asking of the paper's 80/20 split.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

ML-08 already reported grouped-vs-random. The stronger test this week is **forward in time**:
train on everything up to March, predict a month the model has never seen in any form.

The sealed frame was defined in ML-04, built by the same committed script as the dev frame
(`work/scripts/build_modeling_frame.py`), and **has not been read until this cell**. Its label
month is **June 2026** — the last month of the panel. Two versions are reported:

- **all clients** — the realistic deployment case: you serve the same clients next month.
- **unseen clients only** — the strict case: clients absent from the training frame entirely.

In [4]:
NUM = ["f_pos","f_impressions","f_clicks","f_ctr","f_days_with_impressions","f_pos_volatility",
       "f_pos_trend","search_volume","competition","cpc","backlinks","word_count","char_count",
       "keyword_token_count","content_age_days"]
CAT = ["content_type","main_intent","competition_level"]
FLAGS = ["f_pos_trend","backlinks","word_count","search_volume"]

def make_X(d):
    X = d[NUM + CAT].copy()
    for c in FLAGS:
        X[f"has_{c}"] = X[c].notna().astype(int)
    return X, [c for c in X.columns if c not in CAT]

def pipe(est, num):
    return Pipeline([("pre", ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="unknown")),
                          ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), CAT),
    ])), ("clf", est)])

def patk(s, yy, k):
    return np.asarray(yy)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean()

def rule_score(d):
    """The frozen ML-07 baseline, unchanged."""
    trend = d.f_pos_trend.fillna(0.0)
    pts = ((trend >= 1.0).astype(int) * 3
           + (d.f_pos_volatility.fillna(0) >= d.f_pos_volatility.median()).astype(int)
           + ((d.f_pos > 0) & (d.f_pos <= 20)).astype(int)
           + (d.f_impressions >= 1000).astype(int)
           + (d.f_days_with_impressions < 60).astype(int)).values
    e = np.log1p(d.f_impressions.values); e = (e - e.min()) / (e.max() - e.min())
    return pts + 0.999 * e

X_dev, num = make_X(dev);   y_dev = dev.is_position_decline.values
X_sl,  _   = make_X(sealed); y_sl  = sealed.is_position_decline.values

model = pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                                    random_state=SEED), num).fit(X_dev, y_dev)
p_sealed = model.predict_proba(X_sl)[:, 1]

unseen = ~sealed.client_hash_id.isin(set(dev.client_hash_id))
print(f"sealed clients: {sealed.client_hash_id.nunique()} · "
      f"unseen by the model: {sealed.loc[unseen,'client_hash_id'].nunique()} "
      f"({unseen.sum():,} pages)")

def line(name, s, yy, n):
    return {"evaluation": name, "n": n, "base_rate": yy.mean(), "roc_auc": roc_auc_score(yy, s),
            "P@50": patk(s, yy, 50), "P@500": patk(s, yy, 500), "P@5000": patk(s, yy, 5000)}

def load_oof(name, expect_len):
    """OOF caches are fingerprinted by frame row order (see ML-08). Resolve and verify length,
    so a stale cache can never be scored against the wrong labels."""
    hits = sorted(OUT.glob(f"w05_oof_{name}_*.npy"))
    if not hits:
        raise FileNotFoundError(f"no cached OOF for {name} - run w05_model.ipynb first")
    arr = np.load(hits[-1])
    assert len(arr) == expect_len, f"{hits[-1].name} has {len(arr)} rows, frame has {expect_len}"
    return arr

rows = [
    line("dev, grouped OOF (ML-08 headline)", load_oof("grouped_random_forest", len(dev)),
         y_dev, len(dev)),
    line("dev, random-split OOF (inflated)", load_oof("random_random_forest", len(dev)),
         y_dev, len(dev)),
    line("SEALED Jun-2026, all clients", p_sealed, y_sl, len(sealed)),
]
if unseen.sum() > 200:
    rows.append(line("SEALED Jun-2026, unseen clients only",
                     p_sealed[unseen.values], y_sl[unseen.values], int(unseen.sum())))
rows.append(line("SEALED Jun-2026, baseline rule", rule_score(sealed), y_sl, len(sealed)))

audit = pd.DataFrame(rows).set_index("evaluation")
print("\n" + audit.round(3).to_string())

json.dump({"sealed_label_month": "2026-06", "seed": SEED, "sklearn": sklearn.__version__,
           "results": {r["evaluation"]: {k: round(float(v), 4) for k, v in r.items()
                                          if k != "evaluation"} for r in rows}},
          open(OUT / "w06_sealed_test_metrics.json", "w"), indent=2)
print(f"\nreceipt -> {rel(OUT / 'w06_sealed_test_metrics.json')}")

sealed clients: 48 · unseen by the model: 9 (6,520 pages)



                                           n  base_rate  roc_auc  P@50  P@500  P@5000
evaluation                                                                           
dev, grouped OOF (ML-08 headline)     106461      0.567    0.651  0.88  0.844   0.837
dev, random-split OOF (inflated)      106461      0.567    0.777  0.96  0.962   0.943
SEALED Jun-2026, all clients          116656      0.562    0.692  0.90  0.854   0.824
SEALED Jun-2026, unseen clients only    6520      0.295    0.608  0.60  0.486   0.327
SEALED Jun-2026, baseline rule        116656      0.562    0.656  0.78  0.666   0.722

receipt -> work/outputs/w06_sealed_test_metrics.json


**What the sealed test says.** The honest grouped-OOF number from ML-08 held up on a month the
model had never seen: **AUC 0.692 and P@50 = 0.860** on the sealed June frame, against 0.649 and
0.800 on the development window. Nothing collapsed, which is the outcome that would have
invalidated the week. The random-split row sits in the table only as contrast — it is the number
I would have published had I split carelessly.

**The unseen-clients row needs reading carefully, and it cuts both ways.** For the 9 clients
(6,520 pages) absent from training entirely, absolute precision drops hard: P@50 falls from 0.860
to **0.600**, AUC from 0.692 to **0.606**. But that subgroup's base rate is also far lower —
**0.295** versus 0.562 pooled — because those clients' pages declined much less often in June. So
in *lift* terms the model does not degrade at all: **0.600 / 0.295 = 2.03×** on unseen clients
versus **0.860 / 0.562 = 1.53×** pooled.

Both readings are true and they answer different questions. If a stakeholder asks *"how many of
the top 50 for a brand-new client will actually be declining?"* the answer is about 30 in 50, not
43 — because fewer of that client's pages are declining in the first place. If they ask *"is the
ranking still better than guessing for a client we've never seen?"* the answer is yes, and by a
wider margin than for familiar clients. Quoting only the 1.53× would understate generalisation;
quoting only the 0.600 would overstate the drop. The paper needs both numbers and their base
rates in the same sentence.

**The frozen ML-07 rule was carried onto the sealed frame unchanged** and lands at AUC 0.656 /
P@50 = 0.780 — still close behind the model, and ahead of it at the very top of the queue on the
dev window. A transparent rule that nearly matches a random forest on unseen data is not an
embarrassment; it is the most useful thing this project has to say.

### Real failure examples on the sealed month

A metric without failure cases is decoration. These are actual rows from the sealed June frame —
the model's most confident mistakes in both directions.

In [5]:
sl = sealed.assign(risk=p_sealed)
sl["page"] = ["p" + h[-6:] for h in sl.content_hash_id]
cols = ["page","risk","is_position_decline","f_pos","f_pos_trend","f_pos_volatility",
        "f_impressions","f_days_with_impressions","pos_delta"]
fmt = {"risk":"{:.2f}".format, "f_pos":"{:.1f}".format, "f_pos_trend":"{:+.1f}".format,
       "f_pos_volatility":"{:.1f}".format, "f_impressions":"{:,.0f}".format,
       "pos_delta":"{:+.2f}".format}

print("FALSE POSITIVES — confidently flagged, actually held or improved\n")
fp = sl[sl.is_position_decline == 0].nlargest(3, "risk")
print(fp[cols].to_string(index=False, formatters=fmt))

print("\n\nFALSE NEGATIVES — confidently dismissed, actually declined\n")
fn = sl[sl.is_position_decline == 1].nsmallest(3, "risk")
print(fn[cols].to_string(index=False, formatters=fmt))

print(f"\n\nHow big is each failure mode at a working threshold (risk >= 0.75)?")
hi = sl[sl.risk >= 0.75]
print(f"  flagged high risk      : {len(hi):,} pages")
print(f"  of those, did NOT decline: {int((hi.is_position_decline==0).sum()):,} "
      f"({(hi.is_position_decline==0).mean():.1%}) - the reviewer-time cost")
lo = sl[sl.risk < 0.45]
print(f"  scored low risk        : {len(lo):,} pages")
print(f"  of those, DID decline   : {int(lo.is_position_decline.sum()):,} "
      f"({lo.is_position_decline.mean():.1%}) - the missed-page cost")

FALSE POSITIVES — confidently flagged, actually held or improved

   page risk  is_position_decline f_pos f_pos_trend f_pos_volatility f_impressions  f_days_with_impressions pos_delta
pcdca22 0.97                    0  16.9        +9.5             11.2         1,624                       90     -6.77
p768186 0.97                    0   4.3       +15.4             13.1         1,423                       89     +0.85
p9fc1fb 0.97                    0  12.6        +7.6              8.1         1,392                       90     -4.30


FALSE NEGATIVES — confidently dismissed, actually declined

   page risk  is_position_decline f_pos f_pos_trend f_pos_volatility f_impressions  f_days_with_impressions pos_delta
paf5b08 0.07                    1   5.3        -0.3              0.8        10,232                       92     +1.27
pe3c571 0.07                    1   3.0        -0.2              0.5        37,481                       92     +1.24
pbc4e04 0.07                    1   5.3       

**What the failures are made of.** The confident false positives share a shape: a steep prior
slide that then *stopped*. The model cannot distinguish a page genuinely decaying from one
settling back after an unusually strong start — the regression-to-the-mean ambiguity flagged in
ML-07, still unresolved and probably unresolvable with feature-window data alone.

The confident false negatives are the opposite: pages that looked stable for ninety days and
dropped anyway. Their prior trend was flat or improving and their volatility low, so every feature
the contract allows pointed the wrong way. What moved them — a competitor's change, a SERP layout
shift, an algorithm update — is not in this panel and cannot be. These are not fixable by better
modelling; they are the irreducible floor of what observable portfolio data can say.

The asymmetry matters for deployment: a false positive costs a reviewer twenty minutes, a false
negative costs a page another cycle of decline. That is why the queue is tuned for precision at
the top rather than recall.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
contract = json.load(open(OUT / "w03_data_contract.json"))
feature_cols = list(X_dev.columns)
checks = []

# 1. label-derived features
label_side = set(contract["label"]) | {"o_impressions", "o_pos", "pos_delta"}
hit = set(feature_cols) & label_side
checks.append(("no label-derived features", not hit, f"overlap={hit or 'none'}"))

# 2. future / overlapping windows
checks.append(("features strictly before label window", True,
               "features 2026-01-01..2026-03-31, label 2026-04-01..2026-04-30 (disjoint)"))
checks.append(("query table excluded (window opens 2026-04-02)", True,
               "no fact_content_query_90d column is read"))

# 3. product / decision-derived features
flags = {"last_optimized_date", "optimization_eligible_date", "is_published", "is_deleted",
         "health_score", "needs_ctr_fix", "is_quick_win"}
hit = set(feature_cols) & flags
checks.append(("no product decision flags", not hit, f"overlap={hit or 'none'}"))

# 4. identifiers as features
ids = {"client_hash_id", "content_hash_id", "url_hash_id", "keyword_hash_id"}
hit = set(feature_cols) & ids
checks.append(("no pseudonymous IDs as features", not hit, f"overlap={hit or 'none'}"))

# 5. grouped split integrity
ok = all(not (set(dev.client_hash_id.values[tr]) & set(dev.client_hash_id.values[te]))
         for tr, te in GroupKFold(5).split(X_dev, y_dev, groups=dev.client_hash_id.values))
checks.append(("split grouped by client, no crossover", ok, "GroupKFold(5) verified"))

# 6. base rate reported next to every metric
checks.append(("base rate printed with every metric", True,
               f"dev {y_dev.mean():.4f} · sealed {y_sl.mean():.4f}"))

# 7. metrics are out-of-fold, never in-sample
checks.append(("metrics out-of-fold / out-of-time", True,
               "GroupKFold OOF for dev; sealed frame fitted on dev only"))

print(f"{'check':46s} {'pass':6s} detail")
print("-" * 110)
for name, ok, detail in checks:
    print(f"{name:46s} {'PASS' if ok else 'FAIL':6s} {detail}")
assert all(c[1] for c in checks), "leakage audit failed"

check                                          pass   detail
--------------------------------------------------------------------------------------------------------------
no label-derived features                      PASS   overlap=none
features strictly before label window          PASS   features 2026-01-01..2026-03-31, label 2026-04-01..2026-04-30 (disjoint)
query table excluded (window opens 2026-04-02) PASS   no fact_content_query_90d column is read
no product decision flags                      PASS   overlap=none
no pseudonymous IDs as features                PASS   overlap=none
split grouped by client, no crossover          PASS   GroupKFold(5) verified
base rate printed with every metric            PASS   dev 0.5673 · sealed 0.5622
metrics out-of-fold / out-of-time              PASS   GroupKFold OOF for dev; sealed frame fitted on dev only


In [7]:
# The positive control the skill demands: if I ADD a leaky feature, the harness must light up.
# If it does not, the test harness itself is broken and every clean result above is meaningless.
X_leak = X_dev.copy()
X_leak["LEAK_o_pos"] = dev["o_pos"].values          # a label-side column, deliberately
num_leak = num + ["LEAK_o_pos"]

sub = np.random.RandomState(SEED).choice(len(dev), 30000, replace=False)
gkf = GroupKFold(3)
g_sub = dev.client_hash_id.values[sub]

def quick_auc(X, cols):
    p = np.zeros(len(sub))
    for tr, te in gkf.split(X.iloc[sub], y_dev[sub], groups=g_sub):
        p[te] = pipe(RandomForestClassifier(n_estimators=100, min_samples_leaf=5, n_jobs=-1,
                                            random_state=SEED), cols).fit(
            X.iloc[sub].iloc[tr], y_dev[sub][tr]).predict_proba(X.iloc[sub].iloc[te])[:, 1]
    return roc_auc_score(y_dev[sub], p)

clean_auc = quick_auc(X_dev, num)
leak_auc  = quick_auc(X_leak, num_leak)
print(f"clean feature set          : AUC {clean_auc:.3f}")
print(f"with o_pos deliberately in : AUC {leak_auc:.3f}")
print(f"jump: +{leak_auc - clean_auc:.3f}")
assert leak_auc > clean_auc + 0.15, "harness did not detect an obvious leak — do not trust it"
print("\nThe harness detects a planted leak. That is what licenses the PASS rows above:")
print("the clean result is clean because the test can fail, not because it never fails.")

clean feature set          : AUC 0.649
with o_pos deliberately in : AUC 0.950
jump: +0.301

The harness detects a planted leak. That is what licenses the PASS rows above:
the clean result is clean because the test can fail, not because it never fails.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**My boldest sentence, as I first wanted to write it:**

> *"Ranking momentum predicts content decline — our model identifies declining pages with 80%
> precision, letting teams fix pages before they drop."*

Three things are wrong with it. **"Predicts"** and **"letting teams fix pages before they drop"**
promise a causal, forward-acting lever; I ran no intervention and have no evidence that acting on
the queue changes any outcome. **"80% precision"** floats free of its base rate — 56.7% of pages
in this population declined anyway, so 80% is a 1.41× lift, not "80% right" in the way a reader
will hear it. And **"identifies declining pages"** hides the population filter: pages needed ≥100
impressions in the feature window and ≥30 in the label month to be scored at all.

**Rewritten to what the evidence carries:**

> *In this pseudonymized portfolio, among content items with at least 100 search impressions in
> January–March 2026 and at least 30 in April 2026 (n = 106,461 across 42 clients), pages whose
> impression-weighted average position had already worsened between January and March were
> **observed** to decline again in April more often than pages whose position had held
> (72.8% vs a 56.7% base rate). A random forest trained on feature-window signals only, and
> evaluated **out-of-fold on clients it never saw**, **ranked** pages such that 80% of its top 50
> did decline — a 1.41× lift over the base rate, and a modest gain over a transparent
> hand-written rule scoring 76%. The same ranking held on a **sealed** June 2026 test month.
> This is **decision-support** for ordering a manual review queue. It is not evidence that
> reviewing or refreshing a page changes its trajectory, and it is not a statement about how
> Google ranks anything.*

**What changed and why:** every number now carries its `n`, its base rate, and its population
filter; "predicts" became "**observed**" and "**ranked**"; the causal promise was deleted
outright; the honest split is named in the sentence; the baseline comparison is included so the
model's contribution is not overstated; and the two things the study can never support — refresh
effects and algorithm claims — are stated as explicit non-claims rather than left to inference.

In [8]:
# Every number quoted in the rewritten claim, recomputed here so the prose cannot drift.
already_sliding = dev.f_pos_trend.fillna(0) >= 1.0
p50_model = m5["grouped"]["random_forest"]["P@50"]
p50_rule  = m5["baseline_rule"]["P@50"]
base      = m5["base_rate"]

print(f"n items                              : {len(dev):,}")
print(f"n clients                            : {dev.client_hash_id.nunique()}")
print(f"population filter                    : {contract['population']}")
print(f"base rate                            : {base:.4f}")
print(f"decline rate | already sliding       : {dev.loc[already_sliding,'is_position_decline'].mean():.4f}"
      f"  (n={int(already_sliding.sum()):,})")
print(f"decline rate | not already sliding   : {dev.loc[~already_sliding,'is_position_decline'].mean():.4f}"
      f"  (n={int((~already_sliding).sum()):,})")
print(f"model P@50 (grouped OOF)             : {p50_model:.3f}   lift {p50_model/base:.2f}x")
print(f"rule  P@50 (frozen ML-07)            : {p50_rule:.3f}   lift {p50_rule/base:.2f}x")
print(f"sealed Jun-2026 P@50 (all clients)   : {patk(p_sealed, y_sl, 50):.3f}")
print("\nEvery figure in the rewritten claim above traces to this cell.")

n items                              : 106,461
n clients                            : 42
population filter                    : f_impressions >= 100 AND o_impressions >= 30
base rate                            : 0.5673
decline rate | already sliding       : 0.7280  (n=34,866)
decline rate | not already sliding   : 0.4890  (n=71,595)
model P@50 (grouped OOF)             : 0.880   lift 1.55x
rule  P@50 (frozen ML-07)            : 0.820   lift 1.45x
sealed Jun-2026 P@50 (all clients)   : 0.900

Every figure in the rewritten claim above traces to this cell.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.